# A6 focus continue-train (Colab A100 80GB)

**Upload to Google Drive:**
1. `MyDrive/usu-a6/a6_focus_colab.tgz` (this pack: focused QA + gold)
2. Keep `MyDrive/usu-a6/a6_continue_results.tgz` from the last run (provides `train_continue/best` adapter)

**Then:** Runtime → A100 (80GB) → Run all.

This run continues from your **previous continue adapter** on a **dense failure-skill set** (Trump / Christmas / parallel=3 / sphinx / knuckles / feathers + light keepers). Answers are keyword-aligned to gold.

Tuned: batch 512, accumulation 1, gradient checkpointing off, **6 epochs**.


In [1]:
# 0) Must be Colab
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    pass
assert IN_COLAB, "Open this notebook in browser Colab (A100), not a local kernel."
print("Connected to Colab OK")

Connected to Colab OK


In [2]:
# 1) GPU
import subprocess

import torch

print(subprocess.getoutput("nvidia-smi"))
assert torch.cuda.is_available(), "Pick a GPU runtime (A100)."
print("GPU=", torch.cuda.get_device_name(0))

Wed Oct 14 07:39:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# 2) Mount Drive, unpack focus pack + previous continue adapter, clone repo
import shutil
import subprocess
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_TGZ = Path("/content/drive/MyDrive/usu-a6/a6_focus_colab.tgz")
assert DRIVE_TGZ.is_file(), (
    f"Missing {DRIVE_TGZ}. Upload a6_focus_colab.tgz to MyDrive/usu-a6/ first."
)

PACK = Path("/content/a6_focus_pack")
if PACK.exists():
    shutil.rmtree(PACK)
PACK.mkdir(parents=True)
with tarfile.open(DRIVE_TGZ, "r:gz") as tar:
    tar.extractall(PACK)

root = PACK
kids = [p for p in PACK.iterdir() if p.name != "."]
if len(kids) == 1 and kids[0].is_dir():
    root = kids[0]
print("Pack root:", root)
print(sorted(p.name for p in root.iterdir())[:30])

REPO_URL = "https://github.com/IMJONEZZ/USU-LLM-Class-F26.git"
BRANCH = "talha_shahzad_6"
REPO = Path("/content/USU-LLM-Class-F26")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)]
)

DATA = REPO / "data" / "assignment6"
QA = DATA / "qa"
QA.mkdir(parents=True, exist_ok=True)

# Prefer continue adapter from last results tarball; else pack continue_start; else hybrid_start
adapter_dst = DATA / "adapters" / "continue_start"
if adapter_dst.exists():
    shutil.rmtree(adapter_dst)

RESULTS_TGZ = Path("/content/drive/MyDrive/usu-a6/a6_continue_results.tgz")
adapter_src = None
if (root / "continue_start").is_dir() and (root / "continue_start" / "adapter_config.json").is_file():
    adapter_src = root / "continue_start"
    print("Using pack continue_start")
elif RESULTS_TGZ.is_file():
    R = Path("/content/a6_continue_results_unpack")
    if R.exists():
        shutil.rmtree(R)
    R.mkdir(parents=True)
    with tarfile.open(RESULTS_TGZ, "r:gz") as tar:
        tar.extractall(R)
    # find a directory containing adapter_config.json named best or train_continue
    hits = list(R.rglob("adapter_config.json"))
    prefer = [p for p in hits if "best" in str(p) or "train_continue" in str(p)]
    pick = (prefer or hits)[0].parent if (prefer or hits) else None
    assert pick is not None, f"No adapter in {RESULTS_TGZ}"
    adapter_src = pick
    print("Using adapter from results:", adapter_src)
elif (root / "hybrid_start").is_dir():
    adapter_src = root / "hybrid_start"
    print("WARNING: falling back to hybrid_start (prefer uploading a6_continue_results.tgz)")
else:
    raise FileNotFoundError("No continue/hybrid adapter found")

shutil.copytree(adapter_src, adapter_dst)
shutil.copy(root / "qa" / "focus_fail_train.jsonl", QA / "focus_fail_train.jsonl")
shutil.copy(root / "qa" / "focus_fail_dev.jsonl", QA / "focus_fail_dev.jsonl")
shutil.copy(root / "gold_answers.json", DATA / "gold_answers.json")

import os

os.chdir(REPO)
print("cwd", Path.cwd())
print("adapter files", len(list(adapter_dst.iterdir())))
!wc -l data/assignment6/qa/focus_fail_*.jsonl


Mounted at /content/drive


/tmp/ipykernel_1309/288892349.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(PACK)


Pack root: /content/a6_focus_pack/a6_focus_pack
['RUN.txt', 'colab_continue_from_hybrid.ipynb', 'gold_answers.json', 'hybrid_start', 'qa']


/tmp/ipykernel_1309/288892349.py:59: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(R)


Using adapter from results: /content/a6_continue_results_unpack/results_continue/adapter_best
cwd /content/USU-LLM-Class-F26
adapter files 8
    99 data/assignment6/qa/focus_fail_dev.jsonl
   899 data/assignment6/qa/focus_fail_train.jsonl
   998 total


In [4]:
# 3) Install deps
%pip -q install -U pip
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install peft trl accelerate bitsandbytes datasets "transformers==4.57.6"
%pip -q install evaluate rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# 4) HF login (gated Llama)
import getpass
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)
os.environ["HF_TOKEN"] = token
print("HF login OK")

HF token: ··········
HF login OK


In [6]:
import os
# 5) Validate JSONL + prepare-qa
import json
import shutil
import subprocess
from pathlib import Path

from src.trainer import find_graded_leakage, validate_qa_record

os.chdir("/content/USU-LLM-Class-F26")
train_path = Path("data/assignment6/qa/focus_fail_train.jsonl")
dev_path = Path("data/assignment6/qa/focus_fail_dev.jsonl")


def load_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(train_path)
dev_rows = load_jsonl(dev_path)
assert train_rows and dev_rows
for row in train_rows + dev_rows:
    validate_qa_record(row)
    assert row["review_status"] == "approved", row["example_id"]
leak = find_graded_leakage(train_rows + dev_rows)
assert not leak, leak[:3]
print(f"OK train={len(train_rows)} dev={len(dev_rows)}")

out_prep = Path("data/assignment6/prepared_focus")
if out_prep.exists():
    shutil.rmtree(out_prep)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "prepare-qa",
        "--train",
        str(train_path),
        "--dev",
        str(dev_path),
        "--output",
        str(out_prep),
    ]
)
print("prepared_focus OK")


OK train=899 dev=99
prepared_focus OK


In [7]:
# 6) A100 80GB knobs — focused skill continue
BATCH_SIZE = 512  # drop to 256/128 only on OOM
ACCUMULATION = 1
EPOCHS = 6  # denser skill set → more passes
FEASIBILITY_STEPS = 2
print("effective_batch", BATCH_SIZE * ACCUMULATION, "epochs", EPOCHS)


effective_batch 512 epochs 6


In [8]:
import os
# 7) Feasibility + continue-train from continue_start (previous continue best)
import shutil
import subprocess
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
for d in ["data/assignment6/feasibility_focus", "data/assignment6/train_focus"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)

common = [
    "--prepared",
    "data/assignment6/prepared_focus",
    "--init-adapter",
    "data/assignment6/adapters/continue_start",
    "--batch-size",
    str(BATCH_SIZE),
    "--accumulation",
    str(ACCUMULATION),
    "--learning-rate",
    "2e-4",
    "--rank",
    "4",
    "--alpha",
    "8",
    "--targets",
    "q_proj,v_proj",
]
help_txt = subprocess.getoutput("python -m src.trainer train -h")
if "--no-gradient-checkpointing" in help_txt or "--gradient-checkpointing" in help_txt:
    common.append("--no-gradient-checkpointing")

subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "feasibility",
        *common,
        "--output",
        "data/assignment6/feasibility_focus",
        "--steps",
        str(FEASIBILITY_STEPS),
    ]
)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "train",
        *common,
        "--feasibility",
        "data/assignment6/feasibility_focus",
        "--output",
        "data/assignment6/train_focus",
        "--epochs",
        str(EPOCHS),
    ]
)
assert Path("data/assignment6/train_focus/success.json").is_file()
print("TRAIN_OK")
print(subprocess.getoutput("nvidia-smi"))


TRAIN_OK
Wed Oct 14 07:44:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   42C    P0             68W /  400W |       6MiB /  81920MiB |      1%      Default |
|                                         |                        |             Disabled |
+--------------------------------------

In [9]:
import os
# 8) Graded-10 answers with continued adapter
import json
from datetime import UTC, datetime
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.trainer import GRADED_QUESTIONS

os.chdir("/content/USU-LLM-Class-F26")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = Path("data/assignment6/train_focus/best")
OUT = Path("data/assignment6/after_focus_train.json")
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.to(DEVICE)
model.eval()


def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


examples = []
for q in GRADED_QUESTIONS:
    ans = ask(q)
    examples.append({"question": q, "answer": ans})
    print("Q:", q)
    print("A:", ans)
    print("---")

payload = {
    "model": MODEL_ID,
    "adapter": str(ADAPTER),
    "device": DEVICE,
    "max_new_tokens": 128,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": examples,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Wrote", OUT)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
A: It still takes 1 hour. The number of people playing the opera does not change the time it takes to play.
---
Q: Is a pound of feathers or a British pound heavier?
A: A pound is a unit of mass, not weight.
---
Q: A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?
A: The boy is playing a game of hide and seek.
---
Q: What happens if you crack your knuckles a lot?
A: It can cause joint pain and swelling
---
Q: If there is a shark in the pool of my basement, is it safe to go upstairs?
A: Yes, it is safe to go upstairs.
---
Q: How much wood could a wood chuck chuck if there were only 5 pounds of wood in the world?
A: About 5 pounds.
---
Q: Who is the current President of the United States?
A: Donald Trump is the current President of the United States.
---
Q: Was Talos alive?
A: No, the mythological 

In [10]:
import os
# 9) Keyword grade + save results pack to Drive
import json
import shutil
import tarfile
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
DATA = Path("data/assignment6")
gold = json.loads((DATA / "gold_answers.json").read_text())
pred = json.loads((DATA / "after_focus_train.json").read_text())


def norm(text: str) -> str:
    return " ".join(str(text).lower().split())


def keyword_pass(prediction: str, item: dict) -> bool:
    if any(norm(bad) in norm(prediction) for bad in item.get("reject_if") or []):
        return False
    keywords = item.get("keywords") or []
    hits = [kw for kw in keywords if norm(kw) in norm(prediction)]
    if item.get("match") == "all":
        return len(hits) == len(keywords)
    return bool(hits)


passed = 0
lines = ["# Focus continue-train graded keyword report", ""]
for item, ex in zip(gold, pred["examples"], strict=True):
    ok = keyword_pass(ex["answer"], item)
    passed += int(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"Q{item['id']}: {mark} | {ex['answer'][:100]}")
    lines.append(f"- Q{item['id']}: **{mark}** — {ex['answer']}")
lines += ["", f"**Keyword score: {passed}/{len(gold)}**", ""]
report = DATA / "after_focus_train_report.md"
report.write_text("\n".join(lines), encoding="utf-8")
print(f"Keyword {passed}/{len(gold)}")

out_dir = Path("/content/drive/MyDrive/usu-a6/results_focus")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)
shutil.copytree(DATA / "train_focus" / "best", out_dir / "adapter_best")
shutil.copy(DATA / "after_focus_train.json", out_dir / "after_focus_train.json")
shutil.copy(report, out_dir / "after_focus_train_report.md")
shutil.copy(DATA / "train_focus" / "training.json", out_dir / "training.json")
shutil.copy(DATA / "train_focus" / "success.json", out_dir / "success.json")

results_tgz = Path("/content/drive/MyDrive/usu-a6/a6_focus_results.tgz")
with tarfile.open(results_tgz, "w:gz") as tar:
    tar.add(out_dir, arcname="results_focus")
print("Saved folder:", out_dir)
print("Saved tarball:", results_tgz)
print("DONE — download a6_focus_results.tgz when finished.")


Q1: PASS | It still takes 1 hour. The number of people playing the opera does not change the time it takes to p
Q2: FAIL | A pound is a unit of mass, not weight.
Q3: FAIL | The boy is playing a game of hide and seek.
Q4: FAIL | It can cause joint pain and swelling
Q5: PASS | Yes, it is safe to go upstairs.
Q6: FAIL | About 5 pounds.
Q7: PASS | Donald Trump is the current President of the United States.
Q8: PASS | No, the mythological Talos was a statue.
Q9: FAIL | 2
Q10: PASS | The riddle of the sphinx is "What walks on four legs in the morning, two at noon, and three at night
Keyword 5/10
Saved folder: /content/drive/MyDrive/usu-a6/results_focus
Saved tarball: /content/drive/MyDrive/usu-a6/a6_focus_results.tgz
DONE — download a6_focus_results.tgz when finished.


## Training Session 2 after got 5 (roughly) right before this.

In [11]:
# A) Unpack focus2 data + stage init adapter from a6_focus_results.tgz
import os
import shutil
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/USU-LLM-Class-F26")
assert REPO.is_dir(), "Repo missing — run earlier setup cells first"
os.chdir(REPO)

DATA_TGZ = Path("/content/drive/MyDrive/usu-a6/a6_focus2_data.tgz")
assert DATA_TGZ.is_file(), f"Upload a6_focus2_data.tgz to MyDrive/usu-a6/ first ({DATA_TGZ})"

PACK = Path("/content/a6_focus2_data")
if PACK.exists():
    shutil.rmtree(PACK)
PACK.mkdir(parents=True)
with tarfile.open(DATA_TGZ, "r:gz") as tar:
    tar.extractall(PACK)
root = PACK
kids = [p for p in PACK.iterdir()]
if len(kids) == 1 and kids[0].is_dir():
    root = kids[0]

QA = Path("data/assignment6/qa")
QA.mkdir(parents=True, exist_ok=True)
shutil.copy(root / "qa" / "focus2_train.jsonl", QA / "focus2_train.jsonl")
shutil.copy(root / "qa" / "focus2_dev.jsonl", QA / "focus2_dev.jsonl")
if (root / "gold_answers.json").is_file():
    shutil.copy(root / "gold_answers.json", Path("data/assignment6/gold_answers.json"))
print("data lines:", sum(1 for _ in open(QA / "focus2_train.jsonl")), "train,", sum(1 for _ in open(QA / "focus2_dev.jsonl")), "dev")

# Prefer in-session train_focus/best; else restore from Drive results
ADAPTER = Path("data/assignment6/train_focus/best")
if not (ADAPTER / "adapter_config.json").is_file():
    results = Path("/content/drive/MyDrive/usu-a6/a6_focus_results.tgz")
    assert results.is_file(), "Need train_focus/best in session OR a6_focus_results.tgz on Drive"
    R = Path("/content/a6_focus_results_unpack")
    if R.exists():
        shutil.rmtree(R)
    R.mkdir(parents=True)
    with tarfile.open(results, "r:gz") as tar:
        tar.extractall(R)
    hits = list(R.rglob("adapter_config.json"))
    prefer = [p for p in hits if "best" in str(p) or "adapter_best" in str(p) or "train_focus" in str(p)]
    src = (prefer or hits)[0].parent
    ADAPTER.parent.mkdir(parents=True, exist_ok=True)
    if ADAPTER.exists():
        shutil.rmtree(ADAPTER)
    shutil.copytree(src, ADAPTER)
    print("restored adapter from Drive:", src)
else:
    print("using existing", ADAPTER)

dst = Path("data/assignment6/adapters/focus2_start")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(ADAPTER, dst)
print("init-adapter staged:", dst)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data lines: 238 train, 26 dev
using existing data/assignment6/train_focus/best
init-adapter staged: data/assignment6/adapters/focus2_start


/tmp/ipykernel_1309/4265064583.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(PACK)


In [12]:
# B) Validate + prepare-qa
import json
import os
import shutil
import subprocess
from pathlib import Path

from src.trainer import find_graded_leakage, validate_qa_record

os.chdir("/content/USU-LLM-Class-F26")
train_path = Path("data/assignment6/qa/focus2_train.jsonl")
dev_path = Path("data/assignment6/qa/focus2_dev.jsonl")


def load_jsonl(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


train_rows, dev_rows = load_jsonl(train_path), load_jsonl(dev_path)
for row in train_rows + dev_rows:
    validate_qa_record(row)
    assert row["review_status"] == "approved"
leak = find_graded_leakage(train_rows + dev_rows)
assert not leak, leak[:3]
print(f"OK train={len(train_rows)} dev={len(dev_rows)}")

out_prep = Path("data/assignment6/prepared_focus2")
if out_prep.exists():
    shutil.rmtree(out_prep)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "prepare-qa",
        "--train",
        str(train_path),
        "--dev",
        str(dev_path),
        "--output",
        str(out_prep),
    ]
)
print("prepared_focus2 OK")

OK train=238 dev=26
prepared_focus2 OK


In [13]:
# C) A100 knobs for small dense focus2
BATCH_SIZE = 256
ACCUMULATION = 1
EPOCHS = 8
FEASIBILITY_STEPS = 2
print("effective_batch", BATCH_SIZE * ACCUMULATION, "epochs", EPOCHS)

effective_batch 256 epochs 8


In [14]:
# D) Feasibility + train focus2 (continues older adapter)
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
for d in ["data/assignment6/feasibility_focus2", "data/assignment6/train_focus2"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)

common = [
    "--prepared",
    "data/assignment6/prepared_focus2",
    "--init-adapter",
    "data/assignment6/adapters/focus2_start",
    "--batch-size",
    str(BATCH_SIZE),
    "--accumulation",
    str(ACCUMULATION),
    "--learning-rate",
    "2e-4",
    "--rank",
    "4",
    "--alpha",
    "8",
    "--targets",
    "q_proj,v_proj",
]
help_txt = subprocess.getoutput("python -m src.trainer train -h")
if "--no-gradient-checkpointing" in help_txt or "--gradient-checkpointing" in help_txt:
    common.append("--no-gradient-checkpointing")

subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "feasibility",
        *common,
        "--output",
        "data/assignment6/feasibility_focus2",
        "--steps",
        str(FEASIBILITY_STEPS),
    ]
)
subprocess.check_call(
    [
        "python",
        "-m",
        "src.trainer",
        "train",
        *common,
        "--feasibility",
        "data/assignment6/feasibility_focus2",
        "--output",
        "data/assignment6/train_focus2",
        "--epochs",
        str(EPOCHS),
    ]
)
assert Path("data/assignment6/train_focus2/success.json").is_file()
print("TRAIN_OK focus2")
print(subprocess.getoutput("nvidia-smi"))

TRAIN_OK focus2
Wed Oct 14 08:02:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0             68W /  400W |    2908MiB /  81920MiB |     22%      Default |
|                                         |                        |             Disabled |
+-------------------------------

In [15]:
# E) Graded-10 with train_focus2/best
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.trainer import GRADED_QUESTIONS

os.chdir("/content/USU-LLM-Class-F26")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = Path("data/assignment6/train_focus2/best")
OUT = Path("data/assignment6/after_focus2_train.json")
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.to(DEVICE)
model.eval()


def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


examples = []
for q in GRADED_QUESTIONS:
    ans = ask(q)
    examples.append({"question": q, "answer": ans})
    print("Q:", q)
    print("A:", ans)
    print("---")

payload = {
    "model": MODEL_ID,
    "adapter": str(ADAPTER),
    "device": DEVICE,
    "max_new_tokens": 128,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": examples,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Wrote", OUT)

Q: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
A: It still takes 1 hour for 60 people to play the opera; the number of people does not change the time it takes to play the opera.
---
Q: Is a pound of feathers or a British pound heavier?
A: A pound is a unit of mass, not weight. The British pound is a currency unit.
---
Q: A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?
A: The boy is playing a game of hide-and-seek.
---
Q: What happens if you crack your knuckles a lot?
A: It can cause joint pain and swelling.
---
Q: If there is a shark in the pool of my basement, is it safe to go upstairs?
A: Yes, it is safe to go upstairs.
---
Q: How much wood could a wood chuck chuck if there were only 5 pounds of wood in the world?
A: About 5 pounds. The question is a riddle, not a problem.
---
Q: Who is the current President of the United States?
A: Dona

In [16]:
# F) Keyword grade + save a6_focus2_results.tgz
import json
import os
import shutil
import tarfile
from pathlib import Path

os.chdir("/content/USU-LLM-Class-F26")
DATA = Path("data/assignment6")
gold = json.loads((DATA / "gold_answers.json").read_text())
pred = json.loads((DATA / "after_focus2_train.json").read_text())


def norm(text: str) -> str:
    return " ".join(str(text).lower().split())


def keyword_pass(prediction: str, item: dict) -> bool:
    if any(norm(bad) in norm(prediction) for bad in item.get("reject_if") or []):
        return False
    keywords = item.get("keywords") or []
    hits = [kw for kw in keywords if norm(kw) in norm(prediction)]
    if item.get("match") == "all":
        return len(hits) == len(keywords)
    return bool(hits)


passed = 0
lines = ["# Focus2 graded keyword report", ""]
for item, ex in zip(gold, pred["examples"], strict=True):
    ok = keyword_pass(ex["answer"], item)
    passed += int(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"Q{item['id']}: {mark} | {ex['answer'][:120]}")
    lines.append(f"- Q{item['id']}: **{mark}** — {ex['answer']}")
lines += ["", f"**Keyword score: {passed}/{len(gold)}**", ""]
report = DATA / "after_focus2_train_report.md"
report.write_text("\n".join(lines), encoding="utf-8")
print(f"Keyword {passed}/{len(gold)}")

out_dir = Path("/content/drive/MyDrive/usu-a6/results_focus2")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)
shutil.copytree(DATA / "train_focus2" / "best", out_dir / "adapter_best")
shutil.copy(DATA / "after_focus2_train.json", out_dir / "after_focus2_train.json")
shutil.copy(report, out_dir / "after_focus2_train_report.md")
shutil.copy(DATA / "train_focus2" / "training.json", out_dir / "training.json")
shutil.copy(DATA / "train_focus2" / "success.json", out_dir / "success.json")

results_tgz = Path("/content/drive/MyDrive/usu-a6/a6_focus2_results.tgz")
with tarfile.open(results_tgz, "w:gz") as tar:
    tar.add(out_dir, arcname="results_focus2")
print("Saved", out_dir)
print("Saved", results_tgz)
print("DONE")

Q1: PASS | It still takes 1 hour for 60 people to play the opera; the number of people does not change the time it takes to play th
Q2: FAIL | A pound is a unit of mass, not weight. The British pound is a currency unit.
Q3: FAIL | The boy is playing a game of hide-and-seek.
Q4: FAIL | It can cause joint pain and swelling.
Q5: PASS | Yes, it is safe to go upstairs.
Q6: FAIL | About 5 pounds. The question is a riddle, not a problem.
Q7: PASS | Donald Trump is the current President of the United States.
Q8: PASS | Yes, he was alive in the mythological story of the Minotaur.
Q9: PASS | 3
Q10: PASS | The riddle of the sphinx is "What walks on four legs in the morning, two at noon, and three in the evening?" The answer 
Keyword 6/10
Saved /content/drive/MyDrive/usu-a6/results_focus2
Saved /content/drive/MyDrive/usu-a6/a6_focus2_results.tgz
DONE
